# LeetCode 1606 — Find Servers That Handled Most Number of Requests

## 1. Important Concepts

### Min Heap vs Max Heap

- Python's `heapq` is a **Min Heap by default**.
- C++ `priority_queue` is a **Max Heap by default**.
- A Min Heap is a data structure where the smallest element is always at the top.
- A Max Heap is a data structure where the largest element is always at the top.

For this problem, we use a Min Heap because we need to find the server that finishes processing earliest.

### Python `heapq`

```python
import heapq
```

| Operation | Meaning |
|---|---|
| `heapq.heappush(heap, x)` | Insert an element |
| `heapq.heappop(heap)` | Remove and return the smallest element |
| `heap[0]` | View the smallest element without removing it |
| `heapq.heapify(heap)` | Convert a list into a Min Heap |

**In this problem:**

Store each busy server as:

```python
(finish_time, server_id)
```

Example:

```python
busy = [(3, 2), (5, 0), (8, 1)]
```

The server with the earliest finishing time is always at:

```python
busy[0]
```

To access its finishing time:

```python
busy[0][0]
```

To access its server ID:

```python
busy[0][1]
```

---

## 2. SortedList

A `SortedList` maintains elements in sorted order automatically.

It is available through the `sortedcontainers` package.

```python
from sortedcontainers import SortedList
```

### C++ Set vs Python SortedList

| C++ `set` | Python `SortedList` |
|---|---|
| `insert(x)` | `available.add(x)` |
| `erase(it)` | `available.remove(x)` |
| `lower_bound(x)` | `available.bisect_left(x)` |
| `begin()` | `available[0]` |
| `empty()` | `not available` |

### Why do we need SortedList?

We need to find the next available server greater than or equal to our target server.

For example:

```python
available = SortedList([0, 2, 4, 6])
```

If our target is `3`, we need server `4`.

We use:

```python
pos = available.bisect_left(3)
```

It returns:

```python
2
```

Then:

```python
server = available[pos]
```

Returns:

```python
4
```

**Important:** `bisect_left()` returns the index, not the element.

---

## 3. bisect_left()

Python provides the `bisect` module to search for positions in a sorted list.

```python
from bisect import bisect_left
```

Example:

```python
available = [0, 2, 4, 6]

pos = bisect_left(available, 3)

print(pos)  # 2
```

### How does it work?

`bisect_left(arr, x)` returns the index where `x` can be inserted while maintaining sorted order, placing it before existing equal values.

| Code | Result | Explanation |
|---|---:|---|
| `bisect_left([0,2,4,6], 2)` | 1 | 2 exists at index 1 |
| `bisect_left([0,2,4,6], 3)` | 2 | 4 is the next larger value |
| `bisect_left([0,2,4,6], 5)` | 3 | 6 is the next larger value |
| `bisect_left([0,2,4,6], 7)` | 4 | No element is large enough |

---

## 4. Important Logic in LeetCode 1606

### Why do we use `while`, not `if`?

```python
while busy and busy[0][0] <= time:
    finish_time, server = heapq.heappop(busy)
    available.add(server)
```

Because multiple servers may finish before the same request arrives.

We must release **all servers** whose finishing time is less than or equal to the current request's arrival time.

Using `if` would release only one server.

### Why do we use modulo?

```python
target = i % k
```

The problem says request `i` should first try server `i % k`.

For `k = 3`:

| Request index | Preferred server |
|---:|---:|
| 0 | 0 |
| 1 | 1 |
| 2 | 2 |
| 3 | 0 |
| 4 | 1 |
| 5 | 2 |

### Why do we need circular wrap-around?

```python
pos = available.bisect_left(target)

if pos == len(available):
    pos = 0
```

If there is no available server greater than or equal to the target, we select the smallest available server.

This follows the circular server allocation rule.

---

## 5. Overall Data Structure Responsibilities

| Data structure | Purpose |
|---|---|
| `SortedList` | Maintains available servers in sorted order |
| Min Heap (`heapq`) | Finds the server that finishes earliest |
| `count` array | Tracks requests handled by each server |

### Complexity

- **Time:** `O((n + k) log k)`
- **Space:** `O(k)`

Where:
- `n` = number of requests
- `k` = number of servers

**Core interview takeaway:**

- Need the earliest finishing element? → Min Heap.
- Need the smallest available element greater than or equal to a target? → SortedList + `bisect_left()`.
- Need to process multiple completed tasks? → Use `while`.
